# Interactive small-model pilot
Optional 120-second experiment, separate from Gemma. Use one normal Colab session. No remote workers, keepalive tricks or account rotation. CPU is the default. Results are research artifacts and never auto-promoted into the Debian mission.
The notebook trains a tiny byte GRU on host-verified decimal exercises, reports heldout loss and saves its weights. A lower loss is not an income edge.

In [ ]:
import subprocess, sys, json, time, hashlib
from pathlib import Path
lab = Path("/content/v100-interactive-pilot")
if not (lab / ".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch","v100-rtx-reconnect","https://github.com/Mantiee/rlm.git",str(lab)],check=True)
revision = subprocess.check_output(["git","-C",str(lab),"rev-parse","HEAD"],text=True).strip()
print("PINNED SOURCE:", revision)
sys.path.insert(0,str(lab))
import torch
from rlm.v100.insights import verified_record
print("Torch:",torch.__version__)


In [ ]:
# Choose CPU to conserve Colab GPU quota. No Gemma weights are loaded.
DEVICE = "cpu"
STEPS = 20  # 10..100
MAX_SECONDS = 120
assert DEVICE in ("cpu","cuda") and 10 <= STEPS <= 100
assert DEVICE != "cuda" or torch.cuda.is_available()
torch.manual_seed(42)
torch.set_num_threads(2)
rows = [verified_record({"kind":"decimal_calculation","expression":f"{i}*(1+7/1000)*(1-80/10000)-{i}*(1+80/10000)"}) for i in range(101,181)]
# Deterministic disjoint source split; gold answers never appear in input features.
train = [r for r in rows if int(hashlib.sha256(r["group"].encode()).hexdigest(),16)%5]
validation = [r for r in rows if r not in train]
def encode(row):
    prefix = row["messages"][0]["content"]+"\\nassistant:"
    answer = row["messages"][-1]["content"]+"\\n"
    values = [256]+list((prefix+answer).encode())
    labels = [-100]*(1+len(prefix.encode()))+list(answer.encode())
    assert len(values) <= 256
    return torch.tensor([values[:-1]],device=DEVICE),torch.tensor([labels[1:]],device=DEVICE)
class Pilot(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = torch.nn.Embedding(257,32)
        self.gru = torch.nn.GRU(32,64,batch_first=True)
        self.head = torch.nn.Linear(64,257)
    def forward(self,x):
        return self.head(self.gru(self.embed(x))[0])
model = Pilot().to(DEVICE)
print("Parameters:",sum(p.numel() for p in model.parameters()),"Train/validation:",len(train),len(validation))
optimizer = torch.optim.AdamW(model.parameters(),lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)
metrics=[]
started=time.monotonic()
best=float("inf")
output=Path("/content/v100-pilot-results")
output.mkdir(exist_ok=True)
for step in range(STEPS):
    if time.monotonic()-started >= MAX_SECONDS: break
    model.train()
    x,y=encode(train[step%len(train)])
    optimizer.zero_grad()
    loss=loss_fn(model(x).reshape(-1,257),y.reshape(-1))
    assert torch.isfinite(loss)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    model.eval()
    with torch.no_grad():
        values=[]
        for row in validation:
            vx,vy=encode(row)
            values.append(float(loss_fn(model(vx).reshape(-1,257),vy.reshape(-1))))
    measured=sum(values)/len(values)
    record={"step":step+1,"train_loss":float(loss),"heldout_loss":measured,"seconds":time.monotonic()-started}
    metrics.append(record)
    print(json.dumps(record))
    if measured < best:
        best=measured
        # Local trusted checkpoint; do not load pickle checkpoints received from strangers.
        torch.save({"model":model.state_dict(),"optimizer":optimizer.state_dict(),"step":step+1,"rng":torch.get_rng_state()},output/"best-local-checkpoint.pt")
(output/"metrics.json").write_text(json.dumps(metrics,indent=2))
(output/"manifest.json").write_text(json.dumps({"source_revision":revision,"scope":"Tiny separate model; no main-model promotion or profit evidence","device":DEVICE,"steps":len(metrics),"best_heldout_loss":best},indent=2))


In [ ]:
# Save before ending the session; no background connection or quota evasion.
import shutil
archive=shutil.make_archive("/content/v100-pilot-results","zip",output)
from google.colab import files
files.download(archive)
if DEVICE == "cuda":
    del model
    torch.cuda.empty_cache()
print("Pilot complete. Disconnect runtime when finished.")
